# Post Processing

<img src="img_notebook/pp1.png" alt="Descrizione alternativa" width="400" height="300">

## Librerie

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import csv

## Post proc

In [2]:
# 1. CONFIGURAZIONE

# Lunghezza degli step per cui processo i dati (es: ogni 100mm)
STEP = 100

# Cartelle e files -------------------------------------------------------
input_folder = "sorted_data"
output_folder = "postProc_out"
image_folder = os.path.join(output_folder, "plots")
detailed_tables_folder = os.path.join(output_folder, "detailed_tables")
summary_folder = os.path.join(output_folder, "summaries")
doe_file = 'Parameters_DoE.csv'
final_output_file = 'AR_DR_results.csv'

# Creazione directory
os.makedirs(output_folder, exist_ok=True)
os.makedirs(image_folder, exist_ok=True)
os.makedirs(detailed_tables_folder, exist_ok=True)
os.makedirs(summary_folder, exist_ok=True)

# FUNZIONI DI CALCOLO ------------------------------------------------------

def polygon_area(x, y):
    """Calcola l'area di un poligono (Formula di Gauss)."""
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def calculate_vertical_diameter(y_coords):
    """Calcola il diametro VERTICALE come differenza tra Y_max e Y_min."""
    return y_coords.max() - y_coords.min()

# ELABORAZIONE FILE -----------------------------------------------------------

print(f"--- FASE 1: Inizio elaborazione dei file da '{input_folder}' ---")

try:
    file_list = sorted([f for f in os.listdir(input_folder) if f.endswith('.csv')])
    if not file_list:
        raise FileNotFoundError(f"Nessun file .csv trovato in '{input_folder}'.")
except FileNotFoundError as e:
    print(f"ERRORE CRITICO: {e}. Esecuzione interrotta.")
    raise SystemExit()

for file_name in file_list:
    file_path = os.path.join(input_folder, file_name)
    print(f"\nProcessing: {file_name}...")

    # Lettura e preparazione dati
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    df['Z'] = pd.to_numeric(df['Z'], errors='coerce')
    df = df.sort_values(by='Z').reset_index(drop=True)

    # Filtra il DataFrame per tenere solo le Z ogni STEP mm
    df = df[np.isclose(df['Z'] % STEP, 0)].copy()
    # ---------------------------------------------------------------------

    # Controllo se il dataframe è vuoto dopo il filtro
    if df.empty:
        print(f"  -> ATTENZIONE: Nessun dato rimanente per {file_name} dopo il filtraggio per Z ogni 100mm. File saltato.")
        continue # Salta al prossimo file

    print(f"  -> Dati filtrati per Z ogni 100mm. Sezioni rimanenti: {df['Z'].nunique()}")

    df['X_def'] = df['X'] + df['Ux']
    df['Y_def'] = df['Y'] + df['Uy']

    # Calcolo delle aree lungo l'asse Z
    area_deformata_series = df.groupby('Z')[['X_def','Y_def']].apply(lambda g: polygon_area(g['X_def'], g['Y_def']))
    area_indeformata_series = df.groupby('Z')[['X','Y']].apply(lambda g: polygon_area(g['X'], g['Y']))

    # DEFINIZIONE VALORI NOMINALI (INDEFORMATI) 
    Anom = area_indeformata_series.iloc[0]
    first_section_df = df[df['Z'] == df['Z'].iloc[0]]
    Dnom = calculate_vertical_diameter(first_section_df['Y'])

    # CALCOLO AR (Area Reduction)
    Amin = area_deformata_series.min()
    AR_perc = ((Anom - Amin) / Anom) * 100 if Anom > 0 else 0

    # CALCOLO DR (Diameter Reduction) con DIAMETRO VERTICALE
    deformed_vertical_diameters = df.groupby('Z')['Y_def'].apply(calculate_vertical_diameter)
    Dmin = deformed_vertical_diameters.min()
    DR_perc = ((Dnom - Dmin) / Dnom) * 100 if Dnom > 0 else 0
    
    print(f"   -> Calcolato: AR = {AR_perc:.4f}%, DR (Verticale) = {DR_perc:.4f}%")
    print(f"       (Riferimenti: Anom={Anom:.2f}, Dnom_Vert={Dnom:.2f} | Minimi: Amin={Amin:.2f}, Dmin_Vert={Dmin:.2f})")

    # Salvataggio output intermedi ------------------------------------------------------------------------------
    
    # Calcolo diametri
    indeformed_vertical_diameters = df.groupby('Z')['Y'].apply(calculate_vertical_diameter)

    # Trova gli indici dei nodi
    idx_y_max = df.groupby('Z')['Y'].idxmax()
    idx_y_min = df.groupby('Z')['Y'].idxmin()
    idx_x_max = df.groupby('Z')['X'].idxmax()

    # Estrai i valori di spostamento
    Uy_top = df.loc[idx_y_max]['Uy'].values
    Uy_bottom = df.loc[idx_y_min]['Uy'].values
    Ux_max_x = df.loc[idx_x_max]['Ux'].values
    
    # Tabella dettagliata finale ----------------------------------------------------------------------------------
    df_output = pd.DataFrame({
        'Z': area_indeformata_series.index,
        'Area_Indeformata': area_indeformata_series.values,
        'Area_Deformata': area_deformata_series.values,
        'Diametro_Indeformato': indeformed_vertical_diameters.values,
        'Diametro_Deformato': deformed_vertical_diameters.values,
        'Uy_Top_Ymax': Uy_top,
        'Uy_Bottom_Ymin': Uy_bottom,
        'Ux_Max_X': Ux_max_x
    })
        
    output_table_path = os.path.join(detailed_tables_folder, f"detailed_{file_name}")
    df_output.to_csv(output_table_path, index=False, float_format='%.8f')
    print(f"   -> Tabella dettagliata salvata in: {output_table_path}")

    # File di riepilogo ---------------------------------------------------------------------------------------------
    summary_file_path = os.path.join(summary_folder, f"summary_{file_name.replace('.csv', '_AR_DR.csv')}")
    with open(summary_file_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Scenario', 'AR (%)', 'DR (%)'])
        writer.writerow([file_name, AR_perc, DR_perc])

    # GRAFICI ---------------------------------------------------------------------------------------------------------

    # Grafico Aree 
    plt.figure(figsize=(12, 6))
    plt.plot(df_output['Z'], df_output['Area_Indeformata'], marker='.', linestyle='-', color='blue', label=f"Area Indeformata (Anom={Anom:.2f})")
    plt.plot(df_output['Z'], df_output['Area_Deformata'], marker='.', linestyle='--', color='red', label=f"Area Deformata (Amin={Amin:.2f})")
    plt.title(f"Confronto Aree - {file_name}")
    plt.xlabel("Coordinata Z"); plt.ylabel("Area"); plt.legend(); plt.grid(True)
    output_plot_path = os.path.join(image_folder, f"plot_area_{file_name.replace('.csv', '.png')}")
    plt.savefig(output_plot_path)
    plt.close()
    print(f"   -> Grafico AREE salvato in: {output_plot_path}")

    # Grafico Diametri Verticali
    plt.figure(figsize=(12, 6))
    plt.plot(df_output['Z'], df_output['Diametro_Deformato'], marker='.', linestyle='-', color='purple', label=f"Diametro Verticale Deformato (Dmin={Dmin:.2f})")
    plt.axhline(y=Dnom, color='green', linestyle=':', label=f'Diametro Verticale Nominale ({Dnom:.2f})')
    plt.title(f"Andamento Diametro Verticale - {file_name}")
    plt.xlabel("Coordinata Z"); plt.ylabel("Diametro Verticale"); plt.legend(); plt.grid(True)
    output_plot_path = os.path.join(image_folder, f"plot_diam_{file_name.replace('.csv', '.png')}")
    plt.savefig(output_plot_path)
    plt.close()
    print(f"   -> Grafico DIAMETRI salvato in: {output_plot_path}")


# AGGREGAZIONE FINALE con file DoE ---------------------------------------------------------------------------------

print("\n--- FASE 2: Aggregazione finale dei risultati ---")
try:
    doe_df = pd.read_csv(doe_file)
except FileNotFoundError:
    print(f"ERRORE CRITICO: File DoE '{doe_file}' non trovato.")
    raise SystemExit()

summary_files = sorted([f for f in os.listdir(summary_folder) if f.endswith('_AR_DR.csv')])
if not summary_files or (len(doe_df) != len(summary_files)):
    print("AVVISO CRITICO: Il numero di file di riepilogo non corrisponde alle righe del DoE. L'output finale sarà errato.")

results_data_list = []
for s_file in summary_files:
    temp_df = pd.read_csv(os.path.join(summary_folder, s_file))
    results_data_list.append(temp_df.iloc[0])

results_df = pd.DataFrame(results_data_list)
results_df_clean = results_df[['AR (%)', 'DR (%)']]

final_df = pd.concat([doe_df.reset_index(drop=True), results_df_clean.reset_index(drop=True)], axis=1)
final_df.to_csv(final_output_file, index=False, float_format='%.8f')

print(f"\n--- Processo completato. ---")
print(f"La tabella finale aggregata è stata salvata in: '{final_output_file}'")

--- FASE 1: Inizio elaborazione dei file da 'sorted_data' ---

Processing: run_0001_results.csv...
  -> Dati filtrati per Z ogni 100mm. Sezioni rimanenti: 62
   -> Calcolato: AR = 0.0721%, DR (Verticale) = 0.3279%
       (Riferimenti: Anom=756717.26, Dnom_Vert=973.88 | Minimi: Amin=756171.66, Dmin_Vert=970.69)
   -> Tabella dettagliata salvata in: postProc_out/detailed_tables/detailed_run_0001_results.csv
   -> Grafico AREE salvato in: postProc_out/plots/plot_area_run_0001_results.png
   -> Grafico DIAMETRI salvato in: postProc_out/plots/plot_diam_run_0001_results.png

Processing: run_0002_results.csv...
  -> Dati filtrati per Z ogni 100mm. Sezioni rimanenti: 62
   -> Calcolato: AR = 0.1868%, DR (Verticale) = 1.1626%
       (Riferimenti: Anom=756717.26, Dnom_Vert=973.88 | Minimi: Amin=755303.53, Dmin_Vert=962.56)
   -> Tabella dettagliata salvata in: postProc_out/detailed_tables/detailed_run_0002_results.csv
   -> Grafico AREE salvato in: postProc_out/plots/plot_area_run_0002_results.p